<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/07_Proposed_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# 7.1 Environment Setup
# =============================================================================

print("=" * 80)
print("Notebook 07 : Proposed Privacy-Preserving Framework")
print("Environment Setup")
print("=" * 80)

Notebook 07 : Proposed Privacy-Preserving Framework
Environment Setup


In [2]:
# =============================================================================
# Install Required Packages
# =============================================================================

!pip -q install sdv
!pip -q install ctgan
!pip -q install opacus
!pip -q install torchmetrics
!pip -q install scipy
!pip -q install scikit-learn
!pip -q install matplotlib
!pip -q install seaborn
!pip -q install pyyaml
!pip -q install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.9/209.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.5 MB/s eta 0:00:00


In [3]:
# =============================================================================
# Import Libraries
# =============================================================================

import os
import gc
import json
import yaml
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from torch.nn.utils import spectral_norm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.stats import wasserstein_distance
from scipy.stats import ks_2samp

from opacus import PrivacyEngine

warnings.filterwarnings("ignore")

In [4]:
# =============================================================================
# GPU Configuration
# =============================================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU : {torch.cuda.get_device_name(0)}")

    torch.backends.cudnn.benchmark = True

else:

    print("Running on CPU")

Device : cpu
Running on CPU


In [5]:
# =============================================================================
# Random Seed
# =============================================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

print(f"Random Seed : {SEED}")

Random Seed : 42


In [7]:
# =============================================================================
# Google Drive Mount
# =============================================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive Mounted Successfully")

Mounted at /content/drive
Google Drive Mounted Successfully


In [8]:
# =============================================================================
# Performance Configuration
# =============================================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

print("Memory Cleared")

Memory Cleared


In [9]:
# =============================================================================
# 7.2 Project Configuration
# Block 1 : Project Paths
# =============================================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Project")

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

DATASET_DIR = PROJECT_ROOT / "datasets"

SPLIT_DIR = DATASET_DIR / "splits"

RESULT_DIR = PROJECT_ROOT / "results"

MODEL_DIR = RESULT_DIR / "training" / "models"

HISTORY_DIR = RESULT_DIR / "training" / "history"

SYNTHETIC_DIR = RESULT_DIR / "synthetic_data"

print("=" * 80)
print("Project Paths")
print("=" * 80)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Config File  : {CONFIG_PATH}")

Project Paths
Project Root : /content/drive/MyDrive/SPP_GAN_Project
Config File  : /content/drive/MyDrive/SPP_GAN_Project/config.yaml


In [10]:
# =============================================================================
# Block 2 : Load YAML Configuration
# =============================================================================

import yaml

with open(CONFIG_PATH, "r") as file:

    config = yaml.safe_load(file)

print("=" * 80)
print("Configuration Loaded")
print("=" * 80)

for key, value in config.items():

    print(f"{key:<25}: {value}")

Configuration Loaded
batch_size               : 256
beta1                    : 0.5
beta2                    : 0.999
delta                    : 1e-05
device                   : cuda
epochs                   : 300
gradient_clip            : 1.0
latent_dimension         : 128
learning_rate            : 0.0002
noise_multiplier         : 1.1
optimizer                : Adam
privacy_budget           : 4
seed                     : 42


In [11]:
# =============================================================================
# Block 3 : Training Parameters
# =============================================================================

SEED = config["seed"]

DEVICE = torch.device(config["device"])

BATCH_SIZE = config["batch_size"]

EPOCHS = config["epochs"]

LATENT_DIM = config["latent_dimension"]

LEARNING_RATE = config["learning_rate"]

BETA1 = config["beta1"]

BETA2 = config["beta2"]

NOISE_MULTIPLIER = config["noise_multiplier"]

MAX_GRAD_NORM = config["gradient_clip"]

DELTA = config["delta"]

PRIVACY_BUDGET = config["privacy_budget"]

print("=" * 80)
print("Training Parameters")
print("=" * 80)

print(f"Device             : {DEVICE}")
print(f"Epochs             : {EPOCHS}")
print(f"Batch Size         : {BATCH_SIZE}")
print(f"Latent Dimension   : {LATENT_DIM}")
print(f"Learning Rate      : {LEARNING_RATE}")
print(f"Privacy Budget     : {PRIVACY_BUDGET}")

Training Parameters
Device             : cuda
Epochs             : 300
Batch Size         : 256
Latent Dimension   : 128
Learning Rate      : 0.0002
Privacy Budget     : 4


In [12]:
# =============================================================================
# Block 4 : Dataset Paths
# =============================================================================

dataset_paths = {

    "Adult Income": {

        "train": SPLIT_DIR / "Adult Income" / "train.csv",

        "validation": SPLIT_DIR / "Adult Income" / "validation.csv",

        "test": SPLIT_DIR / "Adult Income" / "test.csv"

    },

    "Bank Marketing": {

        "train": SPLIT_DIR / "Bank Marketing" / "train.csv",

        "validation": SPLIT_DIR / "Bank Marketing" / "validation.csv",

        "test": SPLIT_DIR / "Bank Marketing" / "test.csv"

    },

    "Breast Cancer": {

        "train": SPLIT_DIR / "Breast Cancer" / "train.csv",

        "validation": SPLIT_DIR / "Breast Cancer" / "validation.csv",

        "test": SPLIT_DIR / "Breast Cancer" / "test.csv"

    }

}

print("=" * 80)
print("Dataset Paths")
print("=" * 80)

for dataset_name, paths in dataset_paths.items():

    print(f"\n{dataset_name}")

    for split_name, path in paths.items():

        print(f"{split_name:<12}: {path}")

Dataset Paths

Adult Income
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Adult Income/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Adult Income/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Adult Income/test.csv

Bank Marketing
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Bank Marketing/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Bank Marketing/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Bank Marketing/test.csv

Breast Cancer
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Breast Cancer/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Breast Cancer/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/Breast Cancer/test.csv


In [13]:
# =============================================================================
# Block 5 : Verify Project Structure
# =============================================================================

print("=" * 80)
print("Verifying Project Structure")
print("=" * 80)

required_paths = [

    PROJECT_ROOT,

    CONFIG_PATH,

    DATASET_DIR,

    SPLIT_DIR,

    RESULT_DIR

]

for path in required_paths:

    if path.exists():

        print(f"✓ {path}")

    else:

        print(f"✗ Missing : {path}")

Verifying Project Structure
✓ /content/drive/MyDrive/SPP_GAN_Project
✓ /content/drive/MyDrive/SPP_GAN_Project/config.yaml
✓ /content/drive/MyDrive/SPP_GAN_Project/datasets
✓ /content/drive/MyDrive/SPP_GAN_Project/datasets/splits
✓ /content/drive/MyDrive/SPP_GAN_Project/results


In [14]:
# =============================================================================
# 7.3 Load Processed Datasets
# Block 1 : Locate Dataset Folders
# =============================================================================

from pathlib import Path
import pandas as pd

print("=" * 80)
print("Locating Dataset Folders")
print("=" * 80)

SPLIT_DIR = PROJECT_ROOT / "datasets" / "splits"

dataset_paths = {}

for folder in sorted(SPLIT_DIR.iterdir()):

    if folder.is_dir():

        train_file = folder / "train.csv"
        validation_file = folder / "validation.csv"
        test_file = folder / "test.csv"

        if train_file.exists():

            dataset_name = folder.name.replace("_", " ").title()

            dataset_paths[dataset_name] = {

                "train": train_file,

                "validation": validation_file,

                "test": test_file

            }

print(f"Datasets Found : {len(dataset_paths)}\n")

for name in dataset_paths:

    print(name)

Locating Dataset Folders
Datasets Found : 3

Adult Income
Bank Marketing
Breast Cancer


In [15]:
# =============================================================================
# Block 2 : Verify Dataset Paths
# =============================================================================

print("=" * 80)
print("Dataset Paths")
print("=" * 80)

for dataset_name, paths in dataset_paths.items():

    print(f"\n{dataset_name}")

    for split, path in paths.items():

        print(f"{split:<12}: {path}")

Dataset Paths

Adult Income
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/adult_income/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/adult_income/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/adult_income/test.csv

Bank Marketing
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/bank_marketing/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/bank_marketing/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/bank_marketing/test.csv

Breast Cancer
train       : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/breast_cancer/train.csv
validation  : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/breast_cancer/validation.csv
test        : /content/drive/MyDrive/SPP_GAN_Project/datasets/splits/breast_cancer/test.csv


In [16]:
# =============================================================================
# Block 3 : Load Datasets
# =============================================================================

print("=" * 80)
print("Loading Processed Datasets")
print("=" * 80)

datasets = {}

for dataset_name, paths in dataset_paths.items():

    train_df = pd.read_csv(paths["train"])

    validation_df = pd.read_csv(paths["validation"])

    test_df = pd.read_csv(paths["test"])

    datasets[dataset_name] = {

        "train": train_df,

        "validation": validation_df,

        "test": test_df

    }

    print(f"\n{dataset_name}")

    print(f"Train       : {train_df.shape}")

    print(f"Validation  : {validation_df.shape}")

    print(f"Test        : {test_df.shape}")

Loading Processed Datasets

Adult Income
Train       : (22756, 15)
Validation  : (4877, 15)
Test        : (4877, 15)

Bank Marketing
Train       : (31647, 17)
Validation  : (6782, 17)
Test        : (6782, 17)

Breast Cancer
Train       : (398, 32)
Validation  : (85, 32)
Test        : (86, 32)


In [17]:
# =============================================================================
# Block 4 : Dataset Summary
# =============================================================================

print("=" * 80)
print("Dataset Summary")
print("=" * 80)

for dataset_name, data in datasets.items():

    train_df = data["train"]

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(f"Rows            : {train_df.shape[0]}")

    print(f"Columns         : {train_df.shape[1]}")

    print(f"Memory (MB)     : {train_df.memory_usage(deep=True).sum()/1024**2:.2f}")

    print(f"Missing Values  : {train_df.isnull().sum().sum()}")

Dataset Summary

Adult Income
------------------------------------------------------------
Rows            : 22756
Columns         : 15
Memory (MB)     : 2.60
Missing Values  : 0

Bank Marketing
------------------------------------------------------------
Rows            : 31647
Columns         : 17
Memory (MB)     : 4.10
Missing Values  : 0

Breast Cancer
------------------------------------------------------------
Rows            : 398
Columns         : 32
Memory (MB)     : 0.10
Missing Values  : 0


In [18]:
# =============================================================================
# Block 5 : Verify Columns
# =============================================================================

print("=" * 80)
print("Column Verification")
print("=" * 80)

for dataset_name, data in datasets.items():

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(list(data["train"].columns))

Column Verification

Adult Income
------------------------------------------------------------
['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

Bank Marketing
------------------------------------------------------------
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Breast Cancer
------------------------------------------------------------
['id', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture

In [19]:
# =============================================================================
# Block 6 : Dataset Preview
# =============================================================================

for dataset_name, data in datasets.items():

    print("=" * 80)

    print(dataset_name)

    print("=" * 80)

    display(data["train"].head())

Adult Income


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,0.081967,4,0.551719,1.000000,0.478261,4,0.071429,3,4,0,0,0,0.000,0.95122,0
1,0.213115,4,0.370087,0.733333,0.391304,2,0.428571,0,4,1,0,0,0.375,0.95122,1
2,0.081967,4,0.358246,0.000000,0.130435,4,0.857143,1,4,0,0,0,0.000,0.95122,0
3,0.065574,7,0.457402,0.466667,0.652174,4,0.928571,3,4,1,0,0,0.000,0.95122,0
4,0.278689,4,0.062934,0.600000,0.739130,2,0.857143,0,4,1,0,0,0.875,0.95122,1


Bank Marketing


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0.247619,0.636364,1,1,0,0.361910,1,0,0,0.266667,0.454545,0.127527,1,-1,0,3,0
1,0.323810,0.636364,1,1,0,0.397677,1,0,0,0.533333,0.000000,0.315708,1,-1,0,3,0
2,0.114286,0.090909,2,1,0,0.375922,1,0,2,0.500000,0.727273,0.367030,2,-1,0,3,0
3,0.323810,0.090909,1,1,0,0.376475,1,1,2,0.033333,0.545455,0.900467,2,-1,0,3,0
4,0.361905,0.636364,2,1,0,0.381084,0,1,0,0.666667,0.818182,0.306376,2,-1,0,0,0


Breast Cancer


,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,0.415157,0.884711,0.694424,0.963406,0.931265,0.932713,1.000000,1.000000,1.000000,1.000000,...,0.642531,0.875589,0.820732,0.777211,0.982407,0.977849,0.861856,1.000000,0.656172,1
1,0.042865,0.606542,0.282445,0.597946,0.549290,0.487784,0.562608,0.426331,0.455625,0.495562,...,0.324853,0.539638,0.505066,0.603741,0.701247,0.630936,0.680756,0.553588,0.532588,1
2,0.434594,0.447684,0.264914,0.428909,0.369547,0.326268,0.268113,0.150455,0.159899,0.502959,...,0.248532,0.347270,0.269315,0.467687,0.472118,0.327053,0.362199,0.693699,0.608210,0
3,1.000000,0.699712,0.771853,0.681838,0.680166,0.562929,0.455076,0.595588,0.426894,0.144970,...,0.523483,0.511156,0.477096,0.440476,0.253494,0.356843,0.377663,0.014468,0.193321,1
4,0.041164,0.513372,0.269296,0.501229,0.429828,0.729332,0.547314,0.352891,0.457113,0.742604,...,0.202870,0.430820,0.352998,0.469388,0.469616,0.331509,0.480069,0.603845,0.436810,0


In [20]:
# =============================================================================
# Block 7 : Final Verification
# =============================================================================

print("=" * 80)
print("Dataset Verification")
print("=" * 80)

for dataset_name, data in datasets.items():

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(f"Training     : {data['train'].shape}")

    print(f"Validation   : {data['validation'].shape}")

    print(f"Testing      : {data['test'].shape}")

print("\n" + "=" * 80)
print("All Processed Datasets Loaded Successfully")
print("=" * 80)

Dataset Verification

Adult Income
------------------------------------------------------------
Training     : (22756, 15)
Validation   : (4877, 15)
Testing      : (4877, 15)

Bank Marketing
------------------------------------------------------------
Training     : (31647, 17)
Validation   : (6782, 17)
Testing      : (6782, 17)

Breast Cancer
------------------------------------------------------------
Training     : (398, 32)
Validation   : (85, 32)
Testing      : (86, 32)

All Processed Datasets Loaded Successfully


In [21]:
# =============================================================================
# 7.4 Tensor Dataset Preparation
# Block 1 : Imports
# =============================================================================

import numpy as np
import pandas as pd

import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [22]:
# =============================================================================
# Block 2 : Detect Numerical & Categorical Features
# =============================================================================

print("=" * 80)
print("Detecting Feature Types")
print("=" * 80)

feature_info = {}

for dataset_name, data in datasets.items():

    train_df = data["train"]

    categorical_columns = list(

        train_df.select_dtypes(

            include=["object", "category"]

        ).columns

    )

    numerical_columns = [

        col

        for col in train_df.columns

        if col not in categorical_columns

    ]

    feature_info[dataset_name] = {

        "categorical": categorical_columns,

        "numerical": numerical_columns

    }

    print(f"\n{dataset_name}")

    print("-" * 60)

    print("Categorical :", len(categorical_columns))

    print(categorical_columns)

    print("\nNumerical :", len(numerical_columns))

    print(numerical_columns)

Detecting Feature Types

Adult Income
------------------------------------------------------------
Categorical : 0
[]

Numerical : 15
['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']

Bank Marketing
------------------------------------------------------------
Categorical : 0
[]

Numerical : 17
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Breast Cancer
------------------------------------------------------------
Categorical : 0
[]

Numerical : 32
['id', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactn

In [23]:
# =============================================================================
# Block 3 : Label Encoding
# =============================================================================

print("=" * 80)
print("Encoding Categorical Features")
print("=" * 80)

label_encoders = {}

processed_datasets = {}

for dataset_name, data in datasets.items():

    train_df = data["train"].copy()

    validation_df = data["validation"].copy()

    test_df = data["test"].copy()

    encoders = {}

    for column in feature_info[dataset_name]["categorical"]:

        encoder = LabelEncoder()

        combined = pd.concat([

            train_df[column],

            validation_df[column],

            test_df[column]

        ])

        encoder.fit(combined.astype(str))

        train_df[column] = encoder.transform(

            train_df[column].astype(str)

        )

        validation_df[column] = encoder.transform(

            validation_df[column].astype(str)

        )

        test_df[column] = encoder.transform(

            test_df[column].astype(str)

        )

        encoders[column] = encoder

    label_encoders[dataset_name] = encoders

    processed_datasets[dataset_name] = {

        "train": train_df,

        "validation": validation_df,

        "test": test_df

    }

    print(f"{dataset_name:<20} Completed")

Encoding Categorical Features
Adult Income         Completed
Bank Marketing       Completed
Breast Cancer        Completed


In [24]:
# =============================================================================
# Block 4 : Feature Scaling
# =============================================================================

print("=" * 80)
print("Standardizing Numerical Features")
print("=" * 80)

scalers = {}

for dataset_name, data in processed_datasets.items():

    scaler = StandardScaler()

    numerical = feature_info[dataset_name]["numerical"]

    train_df = data["train"]

    validation_df = data["validation"]

    test_df = data["test"]

    train_df[numerical] = scaler.fit_transform(

        train_df[numerical]

    )

    validation_df[numerical] = scaler.transform(

        validation_df[numerical]

    )

    test_df[numerical] = scaler.transform(

        test_df[numerical]

    )

    scalers[dataset_name] = scaler

    print(f"{dataset_name:<20} Standardized")

Standardizing Numerical Features
Adult Income         Standardized
Bank Marketing       Standardized
Breast Cancer        Standardized


In [25]:
# =============================================================================
# Block 5 : Convert To Tensor
# =============================================================================

print("=" * 80)
print("Creating Tensor Datasets")
print("=" * 80)

tensor_datasets = {}

for dataset_name, data in processed_datasets.items():

    train_tensor = torch.tensor(

        data["train"].values,

        dtype=torch.float32

    )

    validation_tensor = torch.tensor(

        data["validation"].values,

        dtype=torch.float32

    )

    test_tensor = torch.tensor(

        data["test"].values,

        dtype=torch.float32

    )

    tensor_datasets[dataset_name] = {

        "train": TensorDataset(train_tensor),

        "validation": TensorDataset(validation_tensor),

        "test": TensorDataset(test_tensor),

        "train_tensor": train_tensor,

        "validation_tensor": validation_tensor,

        "test_tensor": test_tensor

    }

    print(f"\n{dataset_name}")

    print("Train      :", train_tensor.shape)

    print("Validation :", validation_tensor.shape)

    print("Test       :", test_tensor.shape)

Creating Tensor Datasets

Adult Income
Train      : torch.Size([22756, 15])
Validation : torch.Size([4877, 15])
Test       : torch.Size([4877, 15])

Bank Marketing
Train      : torch.Size([31647, 17])
Validation : torch.Size([6782, 17])
Test       : torch.Size([6782, 17])

Breast Cancer
Train      : torch.Size([398, 32])
Validation : torch.Size([85, 32])
Test       : torch.Size([86, 32])


In [26]:
# =============================================================================
# Block 6 : Create DataLoaders
# =============================================================================

print("=" * 80)
print("Creating DataLoaders")
print("=" * 80)

for dataset_name, data in tensor_datasets.items():

    train_loader = DataLoader(

        data["train"],

        batch_size=BATCH_SIZE,

        shuffle=True,

        drop_last=True,

        pin_memory=torch.cuda.is_available()

    )

    validation_loader = DataLoader(

        data["validation"],

        batch_size=BATCH_SIZE,

        shuffle=False,

        drop_last=False,

        pin_memory=torch.cuda.is_available()

    )

    test_loader = DataLoader(

        data["test"],

        batch_size=BATCH_SIZE,

        shuffle=False,

        drop_last=False,

        pin_memory=torch.cuda.is_available()

    )

    data["train_loader"] = train_loader

    data["validation_loader"] = validation_loader

    data["test_loader"] = test_loader

    print(f"{dataset_name:<20} {len(train_loader)} Training Batches")

Creating DataLoaders
Adult Income         88 Training Batches
Bank Marketing       123 Training Batches
Breast Cancer        1 Training Batches


In [27]:
# =============================================================================
# Block 7 : Verification
# =============================================================================

print("=" * 80)
print("Tensor Dataset Verification")
print("=" * 80)

for dataset_name, data in tensor_datasets.items():

    batch = next(iter(data["train_loader"]))[0]

    print(f"\n{dataset_name}")

    print("-" * 60)

    print("Batch Shape :", batch.shape)

    print("Tensor Type :", batch.dtype)

    print("Device      :", batch.device)

print("\n" + "=" * 80)
print("Tensor Dataset Preparation Completed Successfully")
print("=" * 80)

Tensor Dataset Verification

Adult Income
------------------------------------------------------------
Batch Shape : torch.Size([256, 15])
Tensor Type : torch.float32
Device      : cpu

Bank Marketing
------------------------------------------------------------
Batch Shape : torch.Size([256, 17])
Tensor Type : torch.float32
Device      : cpu

Breast Cancer
------------------------------------------------------------
Batch Shape : torch.Size([256, 32])
Tensor Type : torch.float32
Device      : cpu

Tensor Dataset Preparation Completed Successfully


In [28]:
# =============================================================================
# 7.5 Differential Privacy Configuration
# Block 1 : Privacy Parameters
# =============================================================================

print("=" * 80)
print("Differential Privacy Configuration")
print("=" * 80)

# ------------------------------------------------------------------
# Privacy Hyperparameters
# ------------------------------------------------------------------

PRIVACY_BUDGET = config["privacy_budget"]      # epsilon

DELTA = config["delta"]

NOISE_MULTIPLIER = config["noise_multiplier"]

MAX_GRAD_NORM = config["gradient_clip"]

SECURE_MODE = False

print(f"Privacy Budget (ε) : {PRIVACY_BUDGET}")

print(f"Delta (δ)          : {DELTA}")

print(f"Noise Multiplier   : {NOISE_MULTIPLIER}")

print(f"Max Grad Norm      : {MAX_GRAD_NORM}")

print(f"Secure Mode        : {SECURE_MODE}")

Differential Privacy Configuration
Privacy Budget (ε) : 4
Delta (δ)          : 1e-05
Noise Multiplier   : 1.1
Max Grad Norm      : 1.0
Secure Mode        : False


In [29]:
# =============================================================================
# Block 2 : Validate Parameters
# =============================================================================

assert PRIVACY_BUDGET > 0

assert DELTA > 0

assert NOISE_MULTIPLIER > 0

assert MAX_GRAD_NORM > 0

print("=" * 80)

print("Privacy Parameters Valid")

print("=" * 80)

Privacy Parameters Valid


In [30]:
# =============================================================================
# Block 3 : Privacy Configuration Dictionary
# =============================================================================

privacy_config = {

    "epsilon": PRIVACY_BUDGET,

    "delta": DELTA,

    "noise_multiplier": NOISE_MULTIPLIER,

    "max_grad_norm": MAX_GRAD_NORM,

    "secure_mode": SECURE_MODE

}

print("=" * 80)

print("Privacy Configuration")

print("=" * 80)

for key, value in privacy_config.items():

    print(f"{key:<20}: {value}")

Privacy Configuration
epsilon             : 4
delta               : 1e-05
noise_multiplier    : 1.1
max_grad_norm       : 1.0
secure_mode         : False


In [31]:
# =============================================================================
# Block 4 : Privacy Tracking
# =============================================================================

privacy_history = {}

for dataset_name in datasets.keys():

    privacy_history[dataset_name] = {

        "epsilon": [],

        "delta": [],

        "noise_multiplier": [],

        "max_grad_norm": []

    }

print("=" * 80)

print("Privacy Tracking Initialized")

print("=" * 80)

Privacy Tracking Initialized


In [32]:
# =============================================================================
# Block 5 : Configuration Summary
# =============================================================================

print("=" * 80)
print("Differential Privacy Summary")
print("=" * 80)

print(f"Datasets            : {len(datasets)}")

print(f"Training Epochs     : {EPOCHS}")

print(f"Batch Size          : {BATCH_SIZE}")

print(f"Privacy Budget (ε)  : {PRIVACY_BUDGET}")

print(f"Delta (δ)           : {DELTA}")

print(f"Noise Multiplier    : {NOISE_MULTIPLIER}")

print(f"Gradient Clip       : {MAX_GRAD_NORM}")

print("=" * 80)

Differential Privacy Summary
Datasets            : 3
Training Epochs     : 300
Batch Size          : 256
Privacy Budget (ε)  : 4
Delta (δ)           : 1e-05
Noise Multiplier    : 1.1
Gradient Clip       : 1.0


In [33]:
# =============================================================================
# Block 6 : Save Privacy Configuration
# =============================================================================

import json

privacy_config_path = PROJECT_ROOT / "results" / "privacy_config.json"

privacy_config_path.parent.mkdir(

    parents=True,

    exist_ok=True

)

with open(privacy_config_path, "w") as file:

    json.dump(

        privacy_config,

        file,

        indent=4

    )

print("Privacy configuration saved to:")

print(privacy_config_path)

Privacy configuration saved to:
/content/drive/MyDrive/SPP_GAN_Project/results/privacy_config.json


In [34]:
# =============================================================================
# 7.6 Proposed Generator
# Block 1 : Weight Initialization
# =============================================================================

import torch
import torch.nn as nn

def initialize_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [35]:
# =============================================================================
# Block 2 : Residual Block
# =============================================================================

class ResidualBlock(nn.Module):

    def __init__(

        self,

        features,

        dropout=0.20

    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(features, features),

            nn.LayerNorm(features),

            nn.LeakyReLU(0.2, inplace=False),

            nn.Dropout(dropout),

            nn.Linear(features, features),

            nn.LayerNorm(features)

        )

    def forward(self, x):

        return x + self.block(x)

In [36]:
# =============================================================================
# Block 3 : Self Attention
# =============================================================================

class SelfAttention(nn.Module):

    def __init__(

        self,

        features

    ):

        super().__init__()

        self.query = nn.Linear(features, features)

        self.key = nn.Linear(features, features)

        self.value = nn.Linear(features, features)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        q = self.query(x)

        k = self.key(x)

        v = self.value(x)

        attention = self.softmax(

            torch.matmul(

                q,

                k.transpose(-2, -1)

            ) / (x.shape[-1] ** 0.5)

        )

        return torch.matmul(

            attention,

            v

        )

In [37]:
# =============================================================================
# Block 4 : SPP-GAN Generator
# =============================================================================

class ProposedGenerator(nn.Module):

    def __init__(

        self,

        latent_dim,

        output_dim,

        hidden_dims=[256,512,512,256],

        dropout=0.20

    ):

        super().__init__()

        self.input_layer = nn.Sequential(

            nn.Linear(

                latent_dim,

                hidden_dims[0]

            ),

            nn.LayerNorm(

                hidden_dims[0]

            ),

            nn.LeakyReLU(

                0.2,

                inplace=False

            )

        )

        self.residual1 = ResidualBlock(

            hidden_dims[0],

            dropout

        )

        self.expand = nn.Sequential(

            nn.Linear(

                hidden_dims[0],

                hidden_dims[1]

            ),

            nn.LayerNorm(

                hidden_dims[1]

            ),

            nn.LeakyReLU(

                0.2,

                inplace=False

            )

        )

        self.residual2 = ResidualBlock(

            hidden_dims[1],

            dropout

        )

        self.attention = SelfAttention(

            hidden_dims[1]

        )

        self.compress = nn.Sequential(

            nn.Linear(

                hidden_dims[1],

                hidden_dims[3]

            ),

            nn.LayerNorm(

                hidden_dims[3]

            ),

            nn.LeakyReLU(

                0.2,

                inplace=False

            )

        )

        self.output_layer = nn.Sequential(

            nn.Linear(

                hidden_dims[3],

                output_dim

            ),

            nn.Tanh()

        )

        self.apply(

            initialize_weights

        )

    def forward(

        self,

        z

    ):

        x = self.input_layer(z)

        x = self.residual1(x)

        x = self.expand(x)

        x = self.residual2(x)

        x = self.attention(x.unsqueeze(1)).squeeze(1)

        x = self.compress(x)

        return self.output_layer(x)

In [38]:
# =============================================================================
# Block 5 : Model Verification
# =============================================================================

print("=" * 80)
print("Testing Proposed SPP-GAN Generator")
print("=" * 80)

# -------------------------------------------------------------------------
# Device Selection
# -------------------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using Device : {DEVICE}")

if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU          : Not Available (Running on CPU)")

# -------------------------------------------------------------------------
# Model Initialization
# -------------------------------------------------------------------------

OUTPUT_DIM = 31

generator = ProposedGenerator(

    latent_dim=LATENT_DIM,

    output_dim=OUTPUT_DIM

).to(DEVICE)

generator.eval()

# -------------------------------------------------------------------------
# Generate Random Noise
# -------------------------------------------------------------------------

noise = torch.randn(

    8,

    LATENT_DIM,

    device=DEVICE

)

# -------------------------------------------------------------------------
# Forward Pass
# -------------------------------------------------------------------------

with torch.no_grad():

    synthetic = generator(noise)

# -------------------------------------------------------------------------
# Count Parameters
# -------------------------------------------------------------------------

total_params = sum(

    p.numel()

    for p in generator.parameters()

)

trainable_params = sum(

    p.numel()

    for p in generator.parameters()

    if p.requires_grad

)

# -------------------------------------------------------------------------
# Print Summary
# -------------------------------------------------------------------------

print("\nGenerator Verification")
print("-" * 80)

print(f"Noise Shape           : {tuple(noise.shape)}")

print(f"Output Shape          : {tuple(synthetic.shape)}")

print(f"Output Device         : {synthetic.device}")

print(f"Output Min Value      : {synthetic.min().item():.4f}")

print(f"Output Max Value      : {synthetic.max().item():.4f}")

print(f"Contains NaN          : {torch.isnan(synthetic).any().item()}")

print(f"Contains Inf          : {torch.isinf(synthetic).any().item()}")

print(f"Total Parameters      : {total_params:,}")

print(f"Trainable Parameters  : {trainable_params:,}")

print("\n" + "=" * 80)
print("Proposed Generator Verified Successfully")
print("=" * 80)

Testing Proposed SPP-GAN Generator
Using Device : cpu
GPU          : Not Available (Running on CPU)

Generator Verification
--------------------------------------------------------------------------------
Noise Shape           : (8, 128)
Output Shape          : (8, 31)
Output Device         : cpu
Output Min Value      : -0.9969
Output Max Value      : 0.9951
Contains NaN          : False
Contains Inf          : False
Total Parameters      : 1,753,887
Trainable Parameters  : 1,753,887

Proposed Generator Verified Successfully


In [39]:
# =============================================================================
# 7.7 Proposed Discriminator
# Block 1 : Weight Initialization
# =============================================================================

import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm


def initialize_discriminator_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [40]:
# =============================================================================
# Block 2 : Residual Block (Opacus Compatible)
# =============================================================================

class DiscriminatorResidualBlock(nn.Module):

    def __init__(
        self,
        features,
        dropout=0.30
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(
                features,
                features
            ),

            nn.LayerNorm(
                features
            ),

            nn.LeakyReLU(
                negative_slope=0.2,
                inplace=False
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                features,
                features
            ),

            nn.LayerNorm(
                features
            )

        )

        self.activation = nn.LeakyReLU(
            negative_slope=0.2,
            inplace=False
        )

        self.apply(
            initialize_discriminator_weights
        )

    def forward(self, x):

        residual = x

        out = self.block(x)

        out = out + residual

        out = self.activation(out)

        return out

In [41]:
# =============================================================================
# Block 3 : Proposed SPP-GAN Discriminator (Opacus Compatible)
# =============================================================================

class ProposedDiscriminator(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims=[256, 512, 512, 256],
        dropout=0.30
    ):

        super().__init__()

        # ------------------------------------------------------------
        # Input Layer
        # ------------------------------------------------------------

        self.input_layer = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dims[0]
            ),

            nn.LayerNorm(
                hidden_dims[0]
            ),

            nn.LeakyReLU(
                negative_slope=0.2,
                inplace=False
            )

        )

        # ------------------------------------------------------------
        # Residual Block 1
        # ------------------------------------------------------------

        self.residual1 = DiscriminatorResidualBlock(

            features=hidden_dims[0],

            dropout=dropout

        )

        # ------------------------------------------------------------
        # Feature Expansion
        # ------------------------------------------------------------

        self.expand = nn.Sequential(

            nn.Linear(

                hidden_dims[0],

                hidden_dims[1]

            ),

            nn.LayerNorm(

                hidden_dims[1]

            ),

            nn.LeakyReLU(

                negative_slope=0.2,

                inplace=False

            )

        )

        # ------------------------------------------------------------
        # Residual Block 2
        # ------------------------------------------------------------

        self.residual2 = DiscriminatorResidualBlock(

            features=hidden_dims[1],

            dropout=dropout

        )

        # ------------------------------------------------------------
        # Feature Compression
        # ------------------------------------------------------------

        self.compress = nn.Sequential(

            nn.Linear(

                hidden_dims[1],

                hidden_dims[3]

            ),

            nn.LayerNorm(

                hidden_dims[3]

            ),

            nn.LeakyReLU(

                negative_slope=0.2,

                inplace=False

            )

        )

        # ------------------------------------------------------------
        # Output Layer
        # ------------------------------------------------------------

        self.output_layer = nn.Linear(

            hidden_dims[3],

            1

        )

        # ------------------------------------------------------------
        # Initialize Weights
        # ------------------------------------------------------------

        self.apply(

            initialize_discriminator_weights

        )

    def forward(self, x):

        x = self.input_layer(x)

        x = self.residual1(x)

        x = self.expand(x)

        x = self.residual2(x)

        x = self.compress(x)

        x = self.output_layer(x)

        return x

In [42]:
# =============================================================================
# Block 4 : Model Verification
# =============================================================================

print("=" * 80)
print("Testing Proposed SPP-GAN Discriminator")
print("=" * 80)

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

INPUT_DIM = 31

discriminator = ProposedDiscriminator(

    input_dim=INPUT_DIM

).to(DEVICE)

discriminator.eval()

sample = torch.randn(

    8,

    INPUT_DIM,

    device=DEVICE

)

with torch.no_grad():

    output = discriminator(sample)

total_parameters = sum(

    p.numel()

    for p in discriminator.parameters()

)

trainable_parameters = sum(

    p.numel()

    for p in discriminator.parameters()

    if p.requires_grad

)

print(f"Input Shape           : {tuple(sample.shape)}")

print(f"Output Shape          : {tuple(output.shape)}")

print(f"Output Device         : {output.device}")

print(f"Contains NaN          : {torch.isnan(output).any().item()}")

print(f"Contains Inf          : {torch.isinf(output).any().item()}")

print(f"Total Parameters      : {total_parameters:,}")

print(f"Trainable Parameters  : {trainable_parameters:,}")

print("\n" + "=" * 80)
print("Proposed Discriminator Verified Successfully")
print("=" * 80)

Testing Proposed SPP-GAN Discriminator
Input Shape           : (8, 31)
Output Shape          : (8, 1)
Output Device         : cpu
Contains NaN          : False
Contains Inf          : False
Total Parameters      : 933,377
Trainable Parameters  : 933,377

Proposed Discriminator Verified Successfully


In [43]:
# =============================================================================
# 7.8 Attention Block
# Block 1 : Imports
# =============================================================================

import math
import torch
import torch.nn as nn

In [44]:
# =============================================================================
# Block 2 : Multi-Head Self-Attention
# =============================================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(

        self,

        embed_dim,

        num_heads=4,

        dropout=0.10

    ):

        super().__init__()

        assert embed_dim % num_heads == 0, \
            "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim

        self.num_heads = num_heads

        self.head_dim = embed_dim // num_heads

        self.query = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.key = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.value = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.output = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        batch_size = x.size(0)

        seq_length = x.size(1)

        Q = self.query(x)

        K = self.key(x)

        V = self.value(x)

        Q = Q.view(

            batch_size,

            seq_length,

            self.num_heads,

            self.head_dim

        ).transpose(1,2)

        K = K.view(

            batch_size,

            seq_length,

            self.num_heads,

            self.head_dim

        ).transpose(1,2)

        V = V.view(

            batch_size,

            seq_length,

            self.num_heads,

            self.head_dim

        ).transpose(1,2)

        scores = torch.matmul(

            Q,

            K.transpose(-2,-1)

        ) / math.sqrt(self.head_dim)

        attention = torch.softmax(

            scores,

            dim=-1

        )

        attention = self.dropout(attention)

        context = torch.matmul(

            attention,

            V

        )

        context = context.transpose(

            1,

            2

        ).contiguous()

        context = context.view(

            batch_size,

            seq_length,

            self.embed_dim

        )

        output = self.output(context)

        return output

In [45]:
# =============================================================================
# Block 3 : Attention Wrapper
# =============================================================================

class AttentionBlock(nn.Module):

    def __init__(

        self,

        features,

        num_heads=4,

        dropout=0.10

    ):

        super().__init__()

        self.attention = MultiHeadSelfAttention(

            embed_dim=features,

            num_heads=num_heads,

            dropout=dropout

        )

        self.norm = nn.LayerNorm(

            features

        )

    def forward(self, x):

        residual = x

        x = self.attention(x)

        x = self.norm(

            x + residual

        )

        return x

In [46]:
# =============================================================================
# Block 4 : Verify Attention Block
# =============================================================================

print("=" * 80)
print("Testing Multi-Head Attention Block")
print("=" * 80)

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

attention = AttentionBlock(

    features=256,

    num_heads=4

).to(DEVICE)

sample = torch.randn(

    8,

    1,

    256,

    device=DEVICE

)

with torch.no_grad():

    output = attention(sample)

print(f"Input Shape        : {sample.shape}")

print(f"Output Shape       : {output.shape}")

print(f"Device             : {output.device}")

print(f"Contains NaN       : {torch.isnan(output).any().item()}")

print(f"Contains Inf       : {torch.isinf(output).any().item()}")

parameters = sum(

    p.numel()

    for p in attention.parameters()

)

print(f"Parameters         : {parameters:,}")

print("\n" + "=" * 80)
print("Attention Block Verified Successfully")
print("=" * 80)

Testing Multi-Head Attention Block
Input Shape        : torch.Size([8, 1, 256])
Output Shape       : torch.Size([8, 1, 256])
Device             : cpu
Contains NaN       : False
Contains Inf       : False
Parameters         : 263,680

Attention Block Verified Successfully


In [47]:
# =============================================================================
# 7.9 Residual Block
# Block 1 : Imports
# =============================================================================

import torch
import torch.nn as nn

In [48]:
# =============================================================================
# Block 2 : Weight Initialization
# =============================================================================

def initialize_residual_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [49]:
# =============================================================================
# Block 3 : Proposed Residual Block
# =============================================================================

class ResidualBlock(nn.Module):

    def __init__(
        self,
        features,
        dropout=0.20
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(
                features,
                features
            ),

            nn.LayerNorm(
                features
            ),

            nn.LeakyReLU(
                negative_slope=0.2,
                inplace=False
            ),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                features,
                features
            ),

            nn.LayerNorm(
                features
            )

        )

        self.activation = nn.LeakyReLU(
            negative_slope=0.2,
            inplace=False
        )

        self.apply(
            initialize_residual_weights
        )

    def forward(self, x):

        residual = x

        out = self.block(x)

        out = out + residual

        out = self.activation(out)

        return out

In [50]:
# =============================================================================
# Block 4 : Model Verification
# =============================================================================

print("=" * 80)
print("Testing Proposed Residual Block")
print("=" * 80)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FEATURES = 256

residual_block = ResidualBlock(

    features=FEATURES,

    dropout=0.20

).to(DEVICE)

residual_block.eval()

sample = torch.randn(

    8,

    FEATURES,

    device=DEVICE

)

with torch.no_grad():

    output = residual_block(sample)

total_parameters = sum(

    p.numel()

    for p in residual_block.parameters()

)

trainable_parameters = sum(

    p.numel()

    for p in residual_block.parameters()

    if p.requires_grad

)

print(f"Input Shape           : {tuple(sample.shape)}")

print(f"Output Shape          : {tuple(output.shape)}")

print(f"Output Device         : {output.device}")

print(f"Contains NaN          : {torch.isnan(output).any().item()}")

print(f"Contains Inf          : {torch.isinf(output).any().item()}")

print(f"Total Parameters      : {total_parameters:,}")

print(f"Trainable Parameters  : {trainable_parameters:,}")

print("\n" + "=" * 80)
print("Residual Block Verified Successfully")
print("=" * 80)

Testing Proposed Residual Block
Input Shape           : (8, 256)
Output Shape          : (8, 256)
Output Device         : cpu
Contains NaN          : False
Contains Inf          : False
Total Parameters      : 132,608
Trainable Parameters  : 132,608

Residual Block Verified Successfully


In [51]:
# =============================================================================
# 7.11 Training Components
# =============================================================================

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 80)
print("Initializing Training Components")
print("=" * 80)

training_state = {}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

Initializing Training Components


In [52]:
# =============================================================================
# Block 2 : Initialize Models
# =============================================================================

for dataset_name, dataset_info in tensor_datasets.items():

    print(f"\nInitializing : {dataset_name}")

    train_tensor = dataset_info["train_tensor"]

    input_dim = train_tensor.shape[1]

    generator = ProposedGenerator(

        latent_dim=LATENT_DIM,

        output_dim=input_dim

    ).to(DEVICE)

    discriminator = ProposedDiscriminator(

        input_dim=input_dim

    ).to(DEVICE)

    training_state[dataset_name] = {

        "generator": generator,

        "discriminator": discriminator,

        "train_loader": dataset_info["train_loader"],

        "validation_loader": dataset_info["validation_loader"],

        "test_loader": dataset_info["test_loader"],

        "input_dim": input_dim

    }

    print(f"Input Dimension : {input_dim}")


Initializing : Adult Income
Input Dimension : 15

Initializing : Bank Marketing
Input Dimension : 17

Initializing : Breast Cancer
Input Dimension : 32


In [53]:
# =============================================================================
# Block 3 : Optimizers
# =============================================================================

for dataset_name, state in training_state.items():

    optimizer_G = optim.Adam(

        state["generator"].parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

    optimizer_D = optim.Adam(

        state["discriminator"].parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

    state["optimizer_G"] = optimizer_G

    state["optimizer_D"] = optimizer_D

print("Optimizers Initialized")

Optimizers Initialized


In [54]:
# =============================================================================
# 7.11 Training Components
# =============================================================================

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 80)
print("Initializing Training Components")
print("=" * 80)

training_state = {}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

Initializing Training Components


In [55]:
# =============================================================================
# Block 2 : Initialize Models
# =============================================================================

for dataset_name, dataset_info in tensor_datasets.items():

    print(f"\nInitializing : {dataset_name}")

    train_tensor = dataset_info["train_tensor"]

    input_dim = train_tensor.shape[1]

    generator = ProposedGenerator(

        latent_dim=LATENT_DIM,

        output_dim=input_dim

    ).to(DEVICE)

    discriminator = ProposedDiscriminator(

        input_dim=input_dim

    ).to(DEVICE)

    training_state[dataset_name] = {

        "generator": generator,

        "discriminator": discriminator,

        "train_loader": dataset_info["train_loader"],

        "validation_loader": dataset_info["validation_loader"],

        "test_loader": dataset_info["test_loader"],

        "input_dim": input_dim

    }

    print(f"Input Dimension : {input_dim}")


Initializing : Adult Income
Input Dimension : 15

Initializing : Bank Marketing
Input Dimension : 17

Initializing : Breast Cancer
Input Dimension : 32


In [56]:
# =============================================================================
# Block 3 : Optimizers
# =============================================================================

for dataset_name, state in training_state.items():

    optimizer_G = optim.Adam(

        state["generator"].parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

    optimizer_D = optim.Adam(

        state["discriminator"].parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

    state["optimizer_G"] = optimizer_G

    state["optimizer_D"] = optimizer_D

print("Optimizers Initialized")

Optimizers Initialized


In [57]:
# =============================================================================
# Block 4 : Learning Rate Schedulers
# =============================================================================

for dataset_name, state in training_state.items():

    scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(

        state["optimizer_G"],

        mode="min",

        factor=0.5,

        patience=15

    )

    scheduler_D = optim.lr_scheduler.ReduceLROnPlateau(

        state["optimizer_D"],

        mode="min",

        factor=0.5,

        patience=15

    )

    state["scheduler_G"] = scheduler_G

    state["scheduler_D"] = scheduler_D

print("Schedulers Initialized")

Schedulers Initialized


In [58]:
# =============================================================================
# Block 5 : Loss Function
# =============================================================================

criterion = nn.BCEWithLogitsLoss()

for dataset_name in training_state:

    training_state[dataset_name]["criterion"] = criterion

print("Loss Function Initialized")

Loss Function Initialized


In [59]:
# =============================================================================
# Block 6 : Training History
# =============================================================================

for dataset_name in training_state:

    training_state[dataset_name]["history"] = {

        "generator_loss": [],

        "discriminator_loss": [],

        "generator_gradient_norm": [],

        "discriminator_gradient_norm": [],

        "epsilon": []

    }

print("Training History Initialized")

Training History Initialized


In [60]:
# =============================================================================
# Block 6 : Training History
# =============================================================================

for dataset_name in training_state:

    training_state[dataset_name]["history"] = {

        "generator_loss": [],

        "discriminator_loss": [],

        "generator_gradient_norm": [],

        "discriminator_gradient_norm": [],

        "epsilon": []

    }

print("Training History Initialized")

Training History Initialized


In [61]:
# =============================================================================
# Block 7 : Model Summary
# =============================================================================

print("\n" + "=" * 80)
print("Training Components Summary")
print("=" * 80)

for dataset_name, state in training_state.items():

    generator_parameters = sum(

        p.numel()

        for p in state["generator"].parameters()

    )

    discriminator_parameters = sum(

        p.numel()

        for p in state["discriminator"].parameters()

    )

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(f"Input Dimension           : {state['input_dim']}")

    print(f"Generator Parameters      : {generator_parameters:,}")

    print(f"Discriminator Parameters  : {discriminator_parameters:,}")

    print(f"Training Batches          : {len(state['train_loader'])}")


Training Components Summary

Adult Income
------------------------------------------------------------
Input Dimension           : 15
Generator Parameters      : 1,749,775
Discriminator Parameters  : 929,281
Training Batches          : 88

Bank Marketing
------------------------------------------------------------
Input Dimension           : 17
Generator Parameters      : 1,750,289
Discriminator Parameters  : 929,793
Training Batches          : 123

Breast Cancer
------------------------------------------------------------
Input Dimension           : 32
Generator Parameters      : 1,754,144
Discriminator Parameters  : 933,633
Training Batches          : 1


In [62]:
# =============================================================================
# Block 8 : Final Verification
# =============================================================================

print("\n" + "=" * 80)
print("Training Components Ready")
print("=" * 80)

for dataset_name in training_state:

    print(f"✓ {dataset_name}")

print("\nAll models, optimizers, schedulers and dataloaders initialized successfully.")


Training Components Ready
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

All models, optimizers, schedulers and dataloaders initialized successfully.


In [63]:
# =============================================================================
# 7.12 PrivacyEngine Attachment
# Block 1 : Imports
# =============================================================================

from opacus import PrivacyEngine

print("=" * 80)
print("Attaching PrivacyEngine")
print("=" * 80)

Attaching PrivacyEngine


In [64]:
# =============================================================================
# Block 2 : Attach PrivacyEngine
# =============================================================================

for dataset_name, state in training_state.items():

    print(f"\nDataset : {dataset_name}")
    print("-" * 60)

    privacy_engine = PrivacyEngine()

    discriminator_private, optimizer_private, train_loader_private = (
        privacy_engine.make_private(

            module=state["discriminator"],

            optimizer=state["optimizer_D"],

            data_loader=state["train_loader"],

            noise_multiplier=privacy_config["noise_multiplier"],

            max_grad_norm=privacy_config["max_grad_norm"],

            poisson_sampling=False

        )
    )

    # ------------------------------------------------------------
    # Update Training State
    # ------------------------------------------------------------

    state["discriminator"] = discriminator_private

    state["optimizer_D"] = optimizer_private

    state["train_loader"] = train_loader_private

    state["privacy_engine"] = privacy_engine

    print("✓ PrivacyEngine Attached")

    print(f"Noise Multiplier : {privacy_config['noise_multiplier']}")

    print(f"Max Grad Norm    : {privacy_config['max_grad_norm']}")


Dataset : Adult Income
------------------------------------------------------------
✓ PrivacyEngine Attached
Noise Multiplier : 1.1
Max Grad Norm    : 1.0

Dataset : Bank Marketing
------------------------------------------------------------
✓ PrivacyEngine Attached
Noise Multiplier : 1.1
Max Grad Norm    : 1.0

Dataset : Breast Cancer
------------------------------------------------------------
✓ PrivacyEngine Attached
Noise Multiplier : 1.1
Max Grad Norm    : 1.0


In [65]:
# =============================================================================
# Block 3 : Verify Privacy Attachment
# =============================================================================

print("\n" + "=" * 80)
print("Privacy Verification")
print("=" * 80)

for dataset_name, state in training_state.items():

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(
        "Privacy Engine Attached :",
        isinstance(state["privacy_engine"], PrivacyEngine)
    )

    print(
        "Private Model           :",
        state["discriminator"].__class__.__name__
    )

    print(
        "Private Optimizer       :",
        state["optimizer_D"].__class__.__name__
    )

    batch = next(iter(state["train_loader"]))[0]

    print(
        "Batch Shape             :",
        batch.shape
    )


Privacy Verification

Adult Income
------------------------------------------------------------
Privacy Engine Attached : True
Private Model           : GradSampleModule
Private Optimizer       : DPOptimizer
Batch Shape             : torch.Size([256, 15])

Bank Marketing
------------------------------------------------------------
Privacy Engine Attached : True
Private Model           : GradSampleModule
Private Optimizer       : DPOptimizer
Batch Shape             : torch.Size([256, 17])

Breast Cancer
------------------------------------------------------------
Privacy Engine Attached : True
Private Model           : GradSampleModule
Private Optimizer       : DPOptimizer
Batch Shape             : torch.Size([256, 32])


In [66]:
# =============================================================================
# Block 4 : Initialize Privacy Tracking
# =============================================================================

for dataset_name, state in training_state.items():

    if "privacy_history" not in state:

        state["privacy_history"] = {

            "epsilon": [],

            "delta": [],

            "noise_multiplier": [],

            "max_grad_norm": []

        }

print("\nPrivacy Tracking Initialized")


Privacy Tracking Initialized


In [67]:
# =============================================================================
# Block 5 : Summary
# =============================================================================

print("\n" + "=" * 80)
print("PrivacyEngine Successfully Attached")
print("=" * 80)

for dataset_name in training_state:

    print(f"✓ {dataset_name}")

print("\nDifferential Privacy is now enabled for all datasets.")


PrivacyEngine Successfully Attached
✓ Adult Income
✓ Bank Marketing
✓ Breast Cancer

Differential Privacy is now enabled for all datasets.


In [68]:
# =============================================================================
# 7.13 Utility Functions
# Block 1 : Imports
# =============================================================================

import math
import torch
import numpy as np

In [69]:
# =============================================================================
# Block 2 : Noise Generation
# =============================================================================

def generate_noise(
    batch_size,
    latent_dim,
    device
):

    return torch.randn(
        batch_size,
        latent_dim,
        device=device
    )

In [70]:
# =============================================================================
# Block 3 : Label Generation
# =============================================================================

def create_labels(
    batch_size,
    device,
    smooth_real=True
):

    if smooth_real:

        real_labels = torch.empty(
            batch_size,
            1,
            device=device
        ).uniform_(0.9, 1.0)

    else:

        real_labels = torch.ones(
            batch_size,
            1,
            device=device
        )

    fake_labels = torch.zeros(
        batch_size,
        1,
        device=device
    )

    return real_labels, fake_labels

In [71]:
# =============================================================================
# Block 4 : Gradient Norm
# =============================================================================

def gradient_norm(model):

    total_norm = 0.0

    for parameter in model.parameters():

        if parameter.grad is None:

            continue

        norm = parameter.grad.detach().data.norm(2)

        total_norm += norm.item() ** 2

    return math.sqrt(total_norm)

In [72]:
# =============================================================================
# Block 5 : Weight Clipping
# =============================================================================

def clip_weights(
    model,
    clip_value=0.01
):

    for parameter in model.parameters():

        parameter.data.clamp_(

            -clip_value,

            clip_value

        )

In [73]:
# =============================================================================
# Block 6 : Privacy Budget
# =============================================================================

def current_epsilon(

    privacy_engine,

    delta

):

    if privacy_engine is None:

        return 0.0

    return privacy_engine.get_epsilon(delta)

In [74]:
# =============================================================================
# Block 7 : Learning Rate
# =============================================================================

def current_learning_rate(

    optimizer

):

    return optimizer.param_groups[0]["lr"]

In [75]:
# =============================================================================
# Block 8 : Count Parameters
# =============================================================================

def count_parameters(

    model

):

    return sum(

        parameter.numel()

        for parameter in model.parameters()

        if parameter.requires_grad

    )

In [76]:
# =============================================================================
# Block 9 : Model Size
# =============================================================================

def model_size_mb(

    model

):

    parameters = sum(

        parameter.numel()

        for parameter in model.parameters()

    )

    return parameters * 4 / (1024 ** 2)

In [77]:
# =============================================================================
# Block 10 : Device Helper
# =============================================================================

def move_batch_to_device(

    batch,

    device

):

    if isinstance(batch, (list, tuple)):

        batch = batch[0]

    return batch.to(device)

In [78]:
# =============================================================================
# Block 11 : Statistics
# =============================================================================

def tensor_statistics(

    tensor

):

    return {

        "mean": tensor.mean().item(),

        "std": tensor.std().item(),

        "min": tensor.min().item(),

        "max": tensor.max().item()

    }

In [79]:
# =============================================================================
# Block 12 : Save Best Model
# =============================================================================

def save_best_model(

    state,

    generator_loss

):

    if "best_generator_loss" not in state:

        state["best_generator_loss"] = float("inf")

    if generator_loss < state["best_generator_loss"]:

        state["best_generator_loss"] = generator_loss

        state["best_generator"] = {

            key: value.detach().cpu().clone()

            for key, value in state["generator"].state_dict().items()

        }

        return True

    return False

In [80]:
# =============================================================================
# Block 13 : Verification
# =============================================================================

print("=" * 80)
print("Testing Utility Functions")
print("=" * 80)

noise = generate_noise(

    batch_size=8,

    latent_dim=LATENT_DIM,

    device=DEVICE

)

print("Noise Shape :", noise.shape)

real_labels, fake_labels = create_labels(

    batch_size=8,

    device=DEVICE

)

print("Real Labels :", real_labels.shape)

print("Fake Labels :", fake_labels.shape)

print("\nUtility Functions Successfully Initialized")

Testing Utility Functions
Noise Shape : torch.Size([8, 128])
Real Labels : torch.Size([8, 1])
Fake Labels : torch.Size([8, 1])

Utility Functions Successfully Initialized


In [81]:
# =============================================================================
# 7.14 Batch Training
# =============================================================================

def train_batch(
    generator,
    discriminator,
    optimizer_G,
    optimizer_D,
    criterion,
    real_batch,
    latent_dim,
    device,
    privacy_engine=None,
    delta=DELTA
):

    generator.train()
    discriminator.train()

    real_batch = move_batch_to_device(
        real_batch,
        device
    )

    batch_size = real_batch.size(0)

    # ============================================================
    # Create Labels
    # ============================================================

    real_labels, fake_labels = create_labels(

        batch_size=batch_size,

        device=device

    )

    # ============================================================
    # Train Discriminator
    # ============================================================

    optimizer_D.zero_grad(set_to_none=True)

    real_output = discriminator(real_batch)

    d_real_loss = criterion(

        real_output,

        real_labels

    )

    noise = generate_noise(

        batch_size=batch_size,

        latent_dim=latent_dim,

        device=device

    )

    with torch.no_grad():

        fake_samples = generator(noise)

    fake_output = discriminator(

        fake_samples.detach()

    )

    d_fake_loss = criterion(

        fake_output,

        fake_labels

    )

    d_loss = d_real_loss + d_fake_loss

    d_loss.backward()

    optimizer_D.step()

    # ============================================================
    # Train Generator
    # ============================================================

    optimizer_G.zero_grad(set_to_none=True)

    noise = generate_noise(

        batch_size=batch_size,

        latent_dim=latent_dim,

        device=device

    )

    generated_samples = generator(

        noise

    )

    predictions = discriminator(

        generated_samples

    )

    g_loss = criterion(

        predictions,

        real_labels

    )

    g_loss.backward()

    optimizer_G.step()

    # ============================================================
    # Statistics
    # ============================================================

    g_grad = gradient_norm(generator)

    d_grad = gradient_norm(discriminator)

    epsilon = 0.0

    if privacy_engine is not None:

        try:

            epsilon = privacy_engine.get_epsilon(delta)

        except Exception:

            epsilon = 0.0

    return {

        "generator_loss": float(g_loss.item()),

        "discriminator_loss": float(d_loss.item()),

        "generator_gradient_norm": float(g_grad),

        "discriminator_gradient_norm": float(d_grad),

        "epsilon": float(epsilon)

    }

In [82]:
# =============================================================================
# Test Batch Training
# =============================================================================

print("=" * 70)
print("Testing Batch Training")
print("=" * 70)

dataset_name = list(training_state.keys())[0]

state = training_state[dataset_name]

real_batch = next(

    iter(state["train_loader"])

)[0]

stats = train_batch(

    generator=state["generator"],

    discriminator=state["discriminator"],

    optimizer_G=state["optimizer_G"],

    optimizer_D=state["optimizer_D"],

    criterion=state["criterion"],

    real_batch=real_batch,

    latent_dim=LATENT_DIM,

    device=DEVICE,

    privacy_engine=state["privacy_engine"],

    delta=DELTA

)

print("\nBatch Training Successful\n")

for key, value in stats.items():

    print(f"{key:<32}: {value}")

Testing Batch Training

Batch Training Successful

generator_loss                  : 0.9396622180938721
discriminator_loss              : 1.5838439464569092
generator_gradient_norm         : 6.948586587222195
discriminator_gradient_norm     : 15.597056466591106
epsilon                         : 0.17615016305315942


In [83]:
# =============================================================================
# 7.15 Epoch Training
# =============================================================================

def train_epoch(
    generator,
    discriminator,
    optimizer_G,
    optimizer_D,
    criterion,
    train_loader,
    latent_dim,
    device,
    privacy_engine=None,
    delta=DELTA
):

    generator.train()

    discriminator.train()

    # ------------------------------------------------------------
    # Running Statistics
    # ------------------------------------------------------------

    total_g_loss = 0.0

    total_d_loss = 0.0

    total_g_grad = 0.0

    total_d_grad = 0.0

    total_batches = 0

    # ------------------------------------------------------------
    # Iterate Through Batches
    # ------------------------------------------------------------

    for batch in train_loader:

        real_batch = batch[0]

        stats = train_batch(

            generator=generator,

            discriminator=discriminator,

            optimizer_G=optimizer_G,

            optimizer_D=optimizer_D,

            criterion=criterion,

            real_batch=real_batch,

            latent_dim=latent_dim,

            device=device,

            privacy_engine=privacy_engine,

            delta=delta

        )

        total_g_loss += stats["generator_loss"]

        total_d_loss += stats["discriminator_loss"]

        total_g_grad += stats["generator_gradient_norm"]

        total_d_grad += stats["discriminator_gradient_norm"]

        total_batches += 1

    # ------------------------------------------------------------
    # Avoid Division by Zero
    # ------------------------------------------------------------

    if total_batches == 0:

        total_batches = 1

    # ------------------------------------------------------------
    # Epoch Statistics
    # ------------------------------------------------------------

    epoch_stats = {

        "generator_loss":

            total_g_loss / total_batches,

        "discriminator_loss":

            total_d_loss / total_batches,

        "generator_gradient_norm":

            total_g_grad / total_batches,

        "discriminator_gradient_norm":

            total_d_grad / total_batches,

        "epsilon":

            current_epsilon(
                privacy_engine,
                delta
            )

    }

    return epoch_stats

In [84]:
# =============================================================================
# Test Epoch Training
# =============================================================================

print("=" * 80)

print("Testing Epoch Training")

print("=" * 80)

dataset_name = list(training_state.keys())[0]

state = training_state[dataset_name]

epoch_stats = train_epoch(

    generator=state["generator"],

    discriminator=state["discriminator"],

    optimizer_G=state["optimizer_G"],

    optimizer_D=state["optimizer_D"],

    criterion=state["criterion"],

    train_loader=state["train_loader"],

    latent_dim=LATENT_DIM,

    device=DEVICE,

    privacy_engine=state["privacy_engine"],

    delta=DELTA

)

print("\nEpoch Training Successful\n")

for key, value in epoch_stats.items():

    print(f"{key:<32}: {value:.6f}")

Testing Epoch Training

Epoch Training Successful

generator_loss                  : 1.173302
discriminator_loss              : 1.376349
generator_gradient_norm         : 2.478625
discriminator_gradient_norm     : 19.544423
epsilon                         : 0.613311


In [ ]:
# =============================================================================
# 7.16 Full Training
# =============================================================================

import copy
import time

print("=" * 90)
print("Training Proposed SPP-GAN")
print("=" * 90)

training_histories = {}

trained_generators = {}

trained_discriminators = {}

for dataset_name, state in training_state.items():

    print("\n" + "=" * 90)
    print(f"Dataset : {dataset_name}")
    print("=" * 90)

    generator = state["generator"]

    discriminator = state["discriminator"]

    optimizer_G = state["optimizer_G"]

    optimizer_D = state["optimizer_D"]

    criterion = state["criterion"]

    train_loader = state["train_loader"]

    privacy_engine = state["privacy_engine"]

    history = {

        "generator_loss": [],

        "discriminator_loss": [],

        "generator_gradient_norm": [],

        "discriminator_gradient_norm": [],

        "epsilon": [],

        "learning_rate_G": [],

        "learning_rate_D": []

    }

    best_generator_loss = float("inf")

    best_discriminator_loss = float("inf")

    best_generator_state = None

    best_discriminator_state = None

    start_time = time.time()

    # ==========================================================
    # Epoch Loop
    # ==========================================================

    for epoch in range(1, EPOCHS + 1):

        stats = train_epoch(

            generator=generator,

            discriminator=discriminator,

            optimizer_G=optimizer_G,

            optimizer_D=optimizer_D,

            criterion=criterion,

            train_loader=train_loader,

            latent_dim=LATENT_DIM,

            device=DEVICE,

            privacy_engine=privacy_engine,

            delta=DELTA

        )

        # ------------------------------------------------------
        # Store History
        # ------------------------------------------------------

        history["generator_loss"].append(

            stats["generator_loss"]

        )

        history["discriminator_loss"].append(

            stats["discriminator_loss"]

        )

        history["generator_gradient_norm"].append(

            stats["generator_gradient_norm"]

        )

        history["discriminator_gradient_norm"].append(

            stats["discriminator_gradient_norm"]

        )

        history["epsilon"].append(

            stats["epsilon"]

        )

        history["learning_rate_G"].append(

            optimizer_G.param_groups[0]["lr"]

        )

        history["learning_rate_D"].append(

            optimizer_D.param_groups[0]["lr"]

        )

        # ------------------------------------------------------
        # Save Best Generator
        # ------------------------------------------------------

        if stats["generator_loss"] < best_generator_loss:

            best_generator_loss = stats["generator_loss"]

            best_generator_state = copy.deepcopy(

                generator.state_dict()

            )

        # ------------------------------------------------------
        # Save Best Discriminator
        # ------------------------------------------------------

        if stats["discriminator_loss"] < best_discriminator_loss:

            best_discriminator_loss = stats["discriminator_loss"]

            best_discriminator_state = copy.deepcopy(

                discriminator.state_dict()

            )

        # ------------------------------------------------------
        # Progress
        # ------------------------------------------------------

        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:

            print(

                f"Epoch {epoch:03d}/{EPOCHS} | "

                f"G Loss: {stats['generator_loss']:.4f} | "

                f"D Loss: {stats['discriminator_loss']:.4f} | "

                f"Epsilon: {stats['epsilon']:.3f}"

            )

    # ==========================================================
    # Restore Best Models
    # ==========================================================

    generator.load_state_dict(

        best_generator_state

    )

    discriminator.load_state_dict(

        best_discriminator_state

    )

    trained_generators[dataset_name] = generator

    trained_discriminators[dataset_name] = discriminator

    training_histories[dataset_name] = history

    elapsed = time.time() - start_time

    print("\nTraining Completed")

    print(f"Best Generator Loss     : {best_generator_loss:.6f}")

    print(f"Best Discriminator Loss : {best_discriminator_loss:.6f}")

    print(f"Final ε                : {history['epsilon'][-1]:.4f}")

    print(f"Training Time          : {elapsed/60:.2f} minutes")

print("\n" + "=" * 90)
print("All Datasets Successfully Trained")
print("=" * 90)

Training Proposed SPP-GAN

Dataset : Adult Income
Epoch 001/300 | G Loss: 1.4140 | D Loss: 1.2307 | Epsilon: 0.795


In [ ]:
# =============================================================================
# Verify Training
# =============================================================================

print("=" * 90)
print("Training Summary")
print("=" * 90)

for dataset_name in training_histories:

    history = training_histories[dataset_name]

    print(f"\n{dataset_name}")

    print("-" * 60)

    print(f"Epochs Trained           : {len(history['generator_loss'])}")

    print(f"Final Generator Loss     : {history['generator_loss'][-1]:.6f}")

    print(f"Final Discriminator Loss : {history['discriminator_loss'][-1]:.6f}")

    print(f"Final Privacy Budget ε   : {history['epsilon'][-1]:.4f}")

In [ ]:
# =============================================================================
# 7.17 Save Models
# Block 1 : Create Model Directories
# =============================================================================

from pathlib import Path
import json
import torch

print("=" * 80)
print("Creating Model Directories")
print("=" * 80)

MODEL_DIR = PROJECT_ROOT / "results" / "proposed_model"

CHECKPOINT_DIR = MODEL_DIR / "checkpoints"

CONFIG_DIR = MODEL_DIR / "config"

HISTORY_DIR = MODEL_DIR / "history"

for directory in [

    MODEL_DIR,

    CHECKPOINT_DIR,

    CONFIG_DIR,

    HISTORY_DIR

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

print("Directories Created Successfully")

In [ ]:
# =============================================================================
# Block 2 : Save Checkpoints
# =============================================================================

print("=" * 80)
print("Saving Trained Models")
print("=" * 80)

saved_models = {}

for dataset_name, state in training_state.items():

    print(f"\nSaving : {dataset_name}")

    checkpoint = {

        "dataset_name": dataset_name,

        "generator_state_dict":

            state["generator"].state_dict(),

        "discriminator_state_dict":

            state["discriminator"].state_dict(),

        "optimizer_G_state_dict":

            state["optimizer_G"].state_dict(),

        "optimizer_D_state_dict":

            state["optimizer_D"].state_dict(),

        "latent_dim": LATENT_DIM,

        "epochs": EPOCHS,

        "batch_size": BATCH_SIZE,

        "learning_rate": LEARNING_RATE,

        "privacy_budget": PRIVACY_BUDGET,

        "delta": DELTA,

        "noise_multiplier": NOISE_MULTIPLIER,

        "max_grad_norm": MAX_GRAD_NORM

    }

    model_path = CHECKPOINT_DIR / (

        dataset_name.lower()

        .replace(" ", "_")

        + "_spp_gan.pt"

    )

    torch.save(

        checkpoint,

        model_path

    )

    saved_models[dataset_name] = model_path

    print("Saved :", model_path.name)

print("\nAll Models Saved Successfully")

In [ ]:
# =============================================================================
# Block 3 : Save Training History
# =============================================================================

print("=" * 80)
print("Saving Training History")
print("=" * 80)

saved_histories = {}

for dataset_name, history in training_histories.items():

    history_path = HISTORY_DIR / (

        dataset_name.lower()

        .replace(" ", "_")

        + "_history.json"

    )

    with open(

        history_path,

        "w"

    ) as file:

        json.dump(

            history,

            file,

            indent=4

        )

    saved_histories[dataset_name] = history_path

    print("Saved :", history_path.name)

print("\nTraining History Saved Successfully")

In [ ]:
# =============================================================================
# Block 4 : Save Experiment Configuration
# =============================================================================

configuration = {

    "model": "SPP-GAN",

    "datasets": list(training_state.keys()),

    "latent_dimension": LATENT_DIM,

    "epochs": EPOCHS,

    "batch_size": BATCH_SIZE,

    "learning_rate": LEARNING_RATE,

    "privacy_budget": PRIVACY_BUDGET,

    "delta": DELTA,

    "noise_multiplier": NOISE_MULTIPLIER,

    "max_grad_norm": MAX_GRAD_NORM,

    "device": str(DEVICE)

}

config_path = CONFIG_DIR / "experiment_config.json"

with open(

    config_path,

    "w"

) as file:

    json.dump(

        configuration,

        file,

        indent=4

    )

print("=" * 80)
print("Experiment Configuration Saved")
print("=" * 80)

print(config_path)

In [ ]:
# =============================================================================
# Block 5 : Verification
# =============================================================================

print("=" * 80)
print("Saved Files Summary")
print("=" * 80)

for dataset_name in saved_models:

    print(f"\n{dataset_name}")

    print("-" * 50)

    print("Checkpoint")

    print(saved_models[dataset_name])

    print("\nHistory")

    print(saved_histories[dataset_name])

print("\nConfiguration")

print(config_path)

print("\nModel Saving Completed Successfully")

In [ ]:
# =============================================================================
# 7.18 Generate Synthetic Data
# Block 1 : Synthetic Data Generation Function
# =============================================================================

import torch
import pandas as pd


def generate_synthetic_data(
    generator,
    original_dataframe,
    latent_dim,
    device,
    num_samples=None,
    batch_size=1024
):
    """
    Generate synthetic samples using the trained SPP-GAN generator.
    """

    generator.eval()

    if num_samples is None:
        num_samples = len(original_dataframe)

    feature_names = original_dataframe.columns.tolist()

    synthetic_batches = []

    with torch.no_grad():

        for start in range(0, num_samples, batch_size):

            current_batch = min(
                batch_size,
                num_samples - start
            )

            noise = torch.randn(
                current_batch,
                latent_dim,
                device=device
            )

            fake = generator(noise)

            synthetic_batches.append(
                fake.cpu()
            )

    synthetic_tensor = torch.cat(
        synthetic_batches,
        dim=0
    )

    synthetic_df = pd.DataFrame(

        synthetic_tensor.numpy(),

        columns=feature_names

    )

    return synthetic_df

In [ ]:
# =============================================================================
# Block 2 : Generate Synthetic Datasets
# =============================================================================

print("=" * 90)
print("Generating Synthetic Datasets")
print("=" * 90)

synthetic_datasets = {}

generation_statistics = {}

for dataset_name, state in training_state.items():

    print(f"\nGenerating : {dataset_name}")

    train_df = processed_datasets[dataset_name]["train"]

    synthetic_df = generate_synthetic_data(

        generator=state["generator"],

        original_dataframe=train_df,

        latent_dim=LATENT_DIM,

        device=DEVICE,

        num_samples=len(train_df)

    )

    synthetic_datasets[dataset_name] = synthetic_df

    generation_statistics[dataset_name] = {

        "original_samples": len(train_df),

        "synthetic_samples": len(synthetic_df),

        "features": synthetic_df.shape[1]

    }

    print(f"Original Shape  : {train_df.shape}")

    print(f"Synthetic Shape : {synthetic_df.shape}")

In [ ]:
# =============================================================================
# Block 3 : Verify Synthetic Datasets
# =============================================================================

print("=" * 90)
print("Synthetic Dataset Verification")
print("=" * 90)

for dataset_name, synthetic_df in synthetic_datasets.items():

    print(f"\nDataset : {dataset_name}")

    print("-" * 60)

    print("Shape")

    print(synthetic_df.shape)

    print("\nColumns")

    print(list(synthetic_df.columns))

    print("\nFirst Five Samples")

    display(

        synthetic_df.head()

    )

In [ ]:
# =============================================================================
# Block 4 : Generation Summary
# =============================================================================

summary_rows = []

for dataset_name, stats in generation_statistics.items():

    summary_rows.append({

        "Dataset": dataset_name,

        "Original Samples": stats["original_samples"],

        "Synthetic Samples": stats["synthetic_samples"],

        "Features": stats["features"]

    })

generation_summary = pd.DataFrame(summary_rows)

print("=" * 90)
print("Synthetic Data Generation Summary")
print("=" * 90)

display(generation_summary)

In [ ]:
# =============================================================================
# 7.19 Save Synthetic Data
# Block 1 : Create Output Directory
# =============================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 90)
print("Creating Synthetic Data Directories")
print("=" * 90)

SYNTHETIC_DIR = PROJECT_ROOT / "results" / "synthetic_data"

METADATA_DIR = SYNTHETIC_DIR / "metadata"

SYNTHETIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Directories Created Successfully")

In [ ]:
# =============================================================================
# Block 2 : Save Synthetic Datasets
# =============================================================================

print("=" * 90)
print("Saving Synthetic Datasets")
print("=" * 90)

saved_synthetic_files = {}

for dataset_name, synthetic_df in synthetic_datasets.items():

    filename = (

        dataset_name
        .lower()
        .replace(" ", "_")

        + "_spp_gan.csv"

    )

    save_path = SYNTHETIC_DIR / filename

    synthetic_df.to_csv(

        save_path,

        index=False

    )

    saved_synthetic_files[dataset_name] = save_path

    print(f"{dataset_name}")

    print(f"Saved -> {save_path.name}")

In [ ]:
# =============================================================================
# Block 3 : Save Metadata
# =============================================================================

print("=" * 90)
print("Saving Dataset Metadata")
print("=" * 90)

saved_metadata = {}

for dataset_name, synthetic_df in synthetic_datasets.items():

    metadata = {

        "dataset_name":

            dataset_name,

        "model":

            "SPP-GAN",

        "samples":

            int(len(synthetic_df)),

        "features":

            int(synthetic_df.shape[1]),

        "columns":

            list(synthetic_df.columns),

        "latent_dimension":

            LATENT_DIM,

        "epochs":

            EPOCHS,

        "batch_size":

            BATCH_SIZE,

        "learning_rate":

            LEARNING_RATE,

        "privacy_budget":

            PRIVACY_BUDGET,

        "noise_multiplier":

            NOISE_MULTIPLIER,

        "delta":

            DELTA,

        "max_grad_norm":

            MAX_GRAD_NORM

    }

    metadata_path = (

        METADATA_DIR /

        (

            dataset_name

            .lower()

            .replace(" ", "_")

            + "_metadata.json"

        )

    )

    with open(

        metadata_path,

        "w"

    ) as file:

        json.dump(

            metadata,

            file,

            indent=4

        )

    saved_metadata[dataset_name] = metadata_path

    print(

        metadata_path.name

    )

In [ ]:
# =============================================================================
# Block 4 : Save Summary
# =============================================================================

summary_rows = []

for dataset_name in synthetic_datasets:

    df = synthetic_datasets[dataset_name]

    summary_rows.append({

        "Dataset":

            dataset_name,

        "Samples":

            len(df),

        "Features":

            df.shape[1],

        "CSV File":

            saved_synthetic_files[dataset_name].name,

        "Metadata":

            saved_metadata[dataset_name].name

    })

synthetic_summary = pd.DataFrame(

    summary_rows

)

summary_path = SYNTHETIC_DIR / "synthetic_dataset_summary.csv"

synthetic_summary.to_csv(

    summary_path,

    index=False

)

print("\nSummary Saved")

print(summary_path)

In [ ]:
# =============================================================================
# Block 5 : Verification
# =============================================================================

print("=" * 90)
print("Synthetic Data Saved Successfully")
print("=" * 90)

display(

    synthetic_summary

)

print("\nSaved Files")

for dataset_name, path in saved_synthetic_files.items():

    print(

        f"{dataset_name:<20} {path.name}"

    )

print("\nMetadata Files")

for dataset_name, path in saved_metadata.items():

    print(

        f"{dataset_name:<20} {path.name}"

    )

print("\nSummary File")

print(summary_path)

In [ ]:
# =============================================================================
# 7.20 Training History
# Block 1 : Convert Training History
# =============================================================================

import pandas as pd

print("=" * 90)
print("Preparing Training History")
print("=" * 90)

history_dataframes = {}

for dataset_name, history in training_histories.items():

    history_df = pd.DataFrame({

        "Epoch": range(1, len(history["generator_loss"]) + 1),

        "Generator Loss": history["generator_loss"],

        "Discriminator Loss": history["discriminator_loss"],

        "Generator Gradient": history["generator_gradient_norm"],

        "Discriminator Gradient": history["discriminator_gradient_norm"],

        "Privacy Budget (ε)": history["epsilon"],

        "Generator LR": history["learning_rate_G"],

        "Discriminator LR": history["learning_rate_D"]

    })

    history_dataframes[dataset_name] = history_df

    print(f"{dataset_name:<20} {history_df.shape}")

print("\nTraining History Prepared Successfully")

In [ ]:
# =============================================================================
# Block 2 : Save Training History
# =============================================================================

print("=" * 90)
print("Saving Training History")
print("=" * 90)

TRAINING_HISTORY_DIR = PROJECT_ROOT / "results" / "training_history"

TRAINING_HISTORY_DIR.mkdir(

    parents=True,

    exist_ok=True

)

saved_history_files = {}

for dataset_name, history_df in history_dataframes.items():

    filename = (

        dataset_name.lower()

        .replace(" ", "_")

        + "_training_history.csv"

    )

    save_path = TRAINING_HISTORY_DIR / filename

    history_df.to_csv(

        save_path,

        index=False

    )

    saved_history_files[dataset_name] = save_path

    print(filename)

print("\nTraining History Saved Successfully")

In [ ]:
# =============================================================================
# Block 3 : Training Statistics
# =============================================================================

training_statistics = {}

summary_rows = []

for dataset_name, history_df in history_dataframes.items():

    best_epoch = history_df["Generator Loss"].idxmin()

    statistics = {

        "Best Epoch":

            int(best_epoch + 1),

        "Best Generator Loss":

            float(

                history_df["Generator Loss"].min()

            ),

        "Best Discriminator Loss":

            float(

                history_df["Discriminator Loss"].min()

            ),

        "Final Privacy Budget":

            float(

                history_df["Privacy Budget (ε)"].iloc[-1]

            )

    }

    training_statistics[dataset_name] = statistics

    summary_rows.append({

        "Dataset":

            dataset_name,

        **statistics

    })

training_summary = pd.DataFrame(summary_rows)

display(training_summary)

In [ ]:
# =============================================================================
# Block 4 : Plot Training Curves
# =============================================================================

import matplotlib.pyplot as plt

PLOT_DIR = PROJECT_ROOT / "results" / "training_plots"

PLOT_DIR.mkdir(

    parents=True,

    exist_ok=True

)

for dataset_name, history_df in history_dataframes.items():

    # ---------------------------------------------------------
    # Generator / Discriminator Loss
    # ---------------------------------------------------------

    plt.figure(figsize=(10,6))

    plt.plot(

        history_df["Epoch"],

        history_df["Generator Loss"],

        label="Generator"

    )

    plt.plot(

        history_df["Epoch"],

        history_df["Discriminator Loss"],

        label="Discriminator"

    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.title(

        f"{dataset_name} Loss Curves"

    )

    plt.legend()

    plt.grid(True)

    plt.tight_layout()

    plt.savefig(

        PLOT_DIR /

        (

            dataset_name.lower()

            .replace(" ", "_")

            + "_loss.png"

        ),

        dpi=300

    )

    plt.close()

    # ---------------------------------------------------------
    # Privacy Budget
    # ---------------------------------------------------------

    plt.figure(figsize=(10,6))

    plt.plot(

        history_df["Epoch"],

        history_df["Privacy Budget (ε)"]

    )

    plt.xlabel("Epoch")

    plt.ylabel("ε")

    plt.title(

        f"{dataset_name} Privacy Budget"

    )

    plt.grid(True)

    plt.tight_layout()

    plt.savefig(

        PLOT_DIR /

        (

            dataset_name.lower()

            .replace(" ", "_")

            + "_epsilon.png"

        ),

        dpi=300

    )

    plt.close()

print("Training Curves Saved Successfully")

In [ ]:
# =============================================================================
# Block 5 : Verification
# =============================================================================

print("=" * 90)
print("Training History Summary")
print("=" * 90)

display(training_summary)

print("\nSaved History Files")

for dataset_name, path in saved_history_files.items():

    print(f"{dataset_name:<20} {path.name}")

print("\nPlots Saved To")

print(PLOT_DIR)

In [ ]:
# =============================================================================
# 7.21 Model Summary
# Block 1 : Model Information
# =============================================================================

import pandas as pd

print("=" * 90)
print("SPP-GAN MODEL SUMMARY")
print("=" * 90)

summary_rows = []

for dataset_name, state in training_state.items():

    generator = state["generator"]
    discriminator = state["discriminator"]

    generator_parameters = sum(
        p.numel()
        for p in generator.parameters()
    )

    discriminator_parameters = sum(
        p.numel()
        for p in discriminator.parameters()
    )

    summary_rows.append({

        "Dataset": dataset_name,

        "Generator Parameters": generator_parameters,

        "Discriminator Parameters": discriminator_parameters,

        "Total Parameters":
            generator_parameters +
            discriminator_parameters

    })

model_summary = pd.DataFrame(summary_rows)

display(model_summary)

In [ ]:
# =============================================================================
# Block 2 : Training Configuration
# =============================================================================

configuration_summary = pd.DataFrame({

    "Parameter":[

        "Model",

        "Latent Dimension",

        "Batch Size",

        "Epochs",

        "Learning Rate",

        "Noise Multiplier",

        "Privacy Budget (ε)",

        "Delta",

        "Gradient Clipping",

        "Device"

    ],

    "Value":[

        "SPP-GAN",

        LATENT_DIM,

        BATCH_SIZE,

        EPOCHS,

        LEARNING_RATE,

        NOISE_MULTIPLIER,

        PRIVACY_BUDGET,

        DELTA,

        MAX_GRAD_NORM,

        DEVICE

    ]

})

print("=" * 90)
print("Training Configuration")
print("=" * 90)

display(configuration_summary)

In [ ]:
# =============================================================================
# Block 3 : Dataset Summary
# =============================================================================

dataset_rows = []

for dataset_name in processed_datasets.keys():

    train_df = processed_datasets[dataset_name]["train"]

    validation_df = processed_datasets[dataset_name]["validation"]

    test_df = processed_datasets[dataset_name]["test"]

    synthetic_df = synthetic_datasets[dataset_name]

    dataset_rows.append({

        "Dataset":

            dataset_name,

        "Train":

            len(train_df),

        "Validation":

            len(validation_df),

        "Test":

            len(test_df),

        "Synthetic":

            len(synthetic_df),

        "Features":

            train_df.shape[1]

    })

dataset_summary = pd.DataFrame(dataset_rows)

print("=" * 90)
print("Dataset Summary")
print("=" * 90)

display(dataset_summary)

In [ ]:
# =============================================================================
# Block 4 : Training Performance
# =============================================================================

performance_rows = []

for dataset_name, history in training_histories.items():

    performance_rows.append({

        "Dataset":

            dataset_name,

        "Final Generator Loss":

            history["generator_loss"][-1],

        "Final Discriminator Loss":

            history["discriminator_loss"][-1],

        "Final Generator Gradient":

            history["generator_gradient_norm"][-1],

        "Final Discriminator Gradient":

            history["discriminator_gradient_norm"][-1],

        "Final Privacy Budget":

            history["epsilon"][-1]

    })

performance_summary = pd.DataFrame(

    performance_rows

)

print("=" * 90)
print("Training Performance")
print("=" * 90)

display(performance_summary)

In [ ]:
# =============================================================================
# Block 5 : Saved Files
# =============================================================================

saved_files = []

for dataset_name in training_state.keys():

    saved_files.append({

        "Dataset":

            dataset_name,

        "Model":

            saved_models[dataset_name].name,

        "Synthetic Data":

            saved_synthetic_files[dataset_name].name,

        "History":

            saved_history_files[dataset_name].name

    })

saved_files_summary = pd.DataFrame(saved_files)

print("=" * 90)
print("Saved Experiment Files")
print("=" * 90)

display(saved_files_summary)

In [ ]:
# =============================================================================
# Block 6 : Final Summary
# =============================================================================

print("=" * 90)
print("SPP-GAN TRAINING COMPLETED SUCCESSFULLY")
print("=" * 90)

print(f"Datasets Trained      : {len(training_state)}")

print(f"Models Saved          : {len(saved_models)}")

print(f"Synthetic Datasets    : {len(synthetic_datasets)}")

print(f"Training Histories    : {len(training_histories)}")

print(f"Privacy Enabled       : Yes")

print(f"Model                 : SPP-GAN")

print(f"Device                : {DEVICE}")

print("\nNotebook 07 Completed Successfully.")

print("\nReady for Notebook 08 : Comparative Evaluation")

print("=" * 90)